In [1]:
import pandas as pd
import duckdb
import os
import glob

In [2]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

In [3]:
# ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
# ...OR VICE VERSA?

# THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
# THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
# OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
# EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

# TABLES NEEDED (6 out of 12):

    # part_relationships - core table to identify child-child connections
    # parts - to get the part name, for human readability
    # part_categories - allow for more granular analysis by part category
    # inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


#---note: the 'inventory_sets' table is not needed as it just records the quantity of a given part used per set, which is not needed here

In [4]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [5]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [6]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
joined = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT DISTINCT ON(pr.parent_part_num, pr.child_part_num)
                        
                        pc.name AS category,                         --only needs to be shows once, as parent and child likely to have same category 
                    
                        --parent_ps.quantity AS parent_quantity,
                        MIN(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_parent_year,
                        MAX(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_parent_year,
                        --parent_ps.set_num AS parent_set_sum,
                        pr.parent_part_num,
                        parent_p.name AS parent_part_name,
                    
                        child_p.name AS child_part_name,
                        pr.child_part_num,
                        --child_ps.set_num AS child_set_num,
                        MIN(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_child_year,
                        MAX(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_child_year
                        --child_ps.quantity AS child_quantity
                    
                    FROM part_relationships pr
                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num
                    JOIN part_categories pc ON child_p.part_cat_id = pc.id

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R' --only ~8 percent of all parts have this relationship

                    ORDER BY pc.name, pr.parent_part_num, parent_ps.year ASC, child_ps.year ASC --ordering by the years ascending forces the DISTINCT ON to select the minimum
                    


                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [7]:
joined


,category,min_parent_year,max_parent_year,parent_part_num,parent_part_name,child_part_name,child_part_num,min_child_year,max_child_year
0,Animal / Creature Body Parts,2014,2014,11777pr0001,"Animal Body Part, Bird, Eagle Wing - Left with...","Animal Body Part, Bird, Eagle Body with Beak, ...",11435pr0002,2014,2014
1,Animal / Creature Body Parts,2014,2014,11778pr0001,"Animal Body Part, Bird, Eagle Wing - Right wit...","Animal Body Part, Bird, Eagle Wing - Left with...",11777pr0001,2014,2014
2,Animal / Creature Body Parts,2014,2014,11778pr0001,"Animal Body Part, Bird, Eagle Wing - Right wit...","Animal Body Part, Bird, Eagle Body with Beak, ...",11435pr0002,2014,2014
3,Animal / Creature Body Parts,2014,2014,16875pr0001,"Creature Body Part, Dewback Body, Claws and Sh...","Creature Body Part, Dewback Lower Jaw with Tee...",16873pr0001,2014,2014
4,Animal / Creature Body Parts,2014,2014,18153pr0001,"Creature Body Part, Dragon Head Upper Jaw with...","Creature Body Part, Dragon Neck, S-Curve with ...",18234,2014,2014
...,...,...,...,...,...,...,...,...,...
973,Windscreens and Fuselage,1994,2003,4625,Hinge Tile 1 x 4,Windscreen 1 x 4 x 1 1/3 with Bottom Hinge,30161,2002,2003
974,Windscreens and Fuselage,2006,2023,54096,"Slope, Curved 8 x 8 x 2 Double with Cutout",Door 2 x 4 x 6 Curved Aircraft,54097,2006,2023
975,Windscreens and Fuselage,2022,2022,71073,Axle for Spinner,Dome for Ninjago Spinner,69783,2022,2022
976,Windscreens and Fuselage,2012,2021,87613,Aircraft Fuselage Curved Forward 6 x 10 Top,Glass for Aircraft Fuselage Curved Forward 6 x...,87612,2012,2021


In [8]:
#DISCOVERY: ACROSS THE BOARD, THE DATABSSE DOES NOT CONSIDER TWO PARTS THAT STRUCTURALLY COMBINE TO MAKE THE THIRD AS A 'PARENT-CHILD' RELATIONSHIP 
#IN FACT, THERE IS NO RELATIOSNHIP WHATSOEVER. EG. BETWEEN PART 51739, AND 24299 WITH 24307. THIS IS LIKELY BECAUSE THE THE CONNECTIONS UNDERNEATH THE PIECES DIFFER. 
#SO, I WILL HAVE TO USE MY OWN KNOWLEDGE OF PARTS TO PIECE TOGETHER MY ANALYSIS

mydf = duckdb.sql("SELECT * FROM part_relationships WHERE parent_part_num = '51739' OR child_part_num = '51739'")
mydf

┌──────────┬────────────────┬─────────────────┐
│ rel_type │ child_part_num │ parent_part_num │
│ varchar  │    varchar     │     varchar     │
└──────────┴────────────────┴─────────────────┘
                    0 rows                   

In [9]:
#SOME SPECIFIC INSTANCES OF PARENT-CHILD PART RELATIONSHIPS ARE PRESENT AFTER SOME MANUAL INSPECTIONS
#EXTRACT THE MAJORITY BY EXCLUDING WHERE MINIMUM PARENT YEAR = MINIMUM CHILD YEAR...
#...BECAUSE PARTS THAT WERE CREATED IN THE SAME YEAR ARE LIKELY TO BE PART MIRRORINGS, RATHER THAN 'TRUE' PARENT AND CHILD PARTS

mydf = duckdb.sql("SELECT * FROM joined WHERE min_parent_year != min_child_year")
mydf

┌──────────────────────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────┬────────────────┬────────────────┬────────────────┐
│           category           │ min_parent_year │ max_parent_year │ parent_part_num │                               parent_part_name                               │                                 child_part_name                                 │ child_part_num │ min_child_year │ max_child_year │
│           varchar            │      int32      │      int32      │     varchar     │                                   varchar                                    │                                     varchar                                     │    varchar     │     int32      │     int32      │
├──────────────────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────